In [ ]:
!pip install timm==0.9.12 -q  # Vision Transformer models
!pip install albumentations -q

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from PIL import Image
from tqdm.notebook import tqdm
import albumentations as augmentation
from albumentations.pytorch import ToTensorV2
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as models
import timm

# Set random seeds
torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f'🚀 Using device: {device}')

# Path to your uploaded model
model_path = "/kaggle/input/models/aadityakafle/pytorch/default/1/attention_unet_best.pt"


In [ ]:
# === Configuration and Paths ===

# Dataset paths
root_dir = "/kaggle/input/datasets/deathtrooper/multichannel-glaucoma-benchmark-dataset"
FUNDUS_DIR = "/kaggle/input/datasets/deathtrooper/multichannel-glaucoma-benchmark-dataset/full-fundus/full-fundus"

# Load metadata
metadata = pd.read_csv(os.path.join(root_dir, "metadata - standardized.csv"))
metadata = metadata[
    (metadata['types'] != -1)
]
# metadata = metadata[
#     ~metadata['fundus_oc_seg'].isnull() &
#     (metadata['fundus_oc_seg'] != 'Not Visible') &
#     ~metadata['fundus_od_seg'].isnull()
# ].reset_index(drop=True)

In [ ]:
metadata['types'].unique()

# Preprocessing

In [ ]:
# === 1. Imports ===

import cv2
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import argrelextrema
from scipy.stats import mode
from skimage import measure

In [ ]:
# === 2. Utility Functions ===


def show(img, title=""):
    plt.figure(figsize=(4,4))
    if len(img.shape) == 2:
        plt.imshow(img, cmap='gray')
    else:
        plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        plt.title(title)
        plt.axis('off')

In [ ]:
# === 3. Clahe Preprocessing (Paper Section III) ===

# why the cliplimit of 2??

def apply_clahe(img):
    """
    Apply CLAHE on the green channel (standard practice for fundus images)
    """
    green = img[:, :, 1]
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    enhanced = clahe.apply(green)
    return enhanced

In [ ]:
# === 4. Dynamic Histogram-Based Threshold (Paer Section IV) ===

def compute_dynamic_threshold(gray):
    """
    Approximation of paper's histogram-extrema-based thresholding
    """
    hist = cv2.calcHist([gray],[0],None,[256],[0,256]).flatten()
    hist = np.log1p(hist) # log normalization
    
    
    # smooth histogram
    hist_smooth = cv2.GaussianBlur(hist, (9,1), 0).flatten()
    
    
    # find local minima and maxima
    local_max = argrelextrema(hist_smooth, np.greater)[0]
    local_min = argrelextrema(hist_smooth, np.less)[0]
    
    
    # restrict search space (paper uses t1=192, t2=128)
    local_max = [x for x in local_max if x <= 192]
    local_min = [x for x in local_min if x <= 128]
    
    
    best_thresh = 0
    best_score = 0
    
    
    for m1 in local_max:
        for mn in local_min:
            for m2 in local_max:
                if m1 < mn < m2:
                    score = abs(hist_smooth[m1] - hist_smooth[mn]) \
                    + abs(hist_smooth[mn] - hist_smooth[m2])
                    if score > best_score:
                        best_score = score
                        best_thresh = mn
    
    
    return best_thresh

In [ ]:
# === 5. Foreground Segmentation ===

def segment_foreground(gray):
    thresh = compute_dynamic_threshold(gray)
    _, mask = cv2.threshold(gray, thresh, 255, cv2.THRESH_BINARY)
    
    
    # keep largest connected component
    labels = measure.label(mask)
    props = measure.regionprops(labels)
    if len(props) == 0:
        return mask
    
    
    largest = max(props, key=lambda x: x.area)
    clean_mask = np.zeros_like(mask)
    clean_mask[labels == largest.label] = 255
    
    
    return clean_mask

In [ ]:
# === 6. Fundus Center Estimation (Paper Algorithm 2) ===

def estimate_center(mask):
    h, w = mask.shape
    row_medians = []
    col_medians = []
    
    
    for i in range(h):
        cols = np.where(mask[i] > 0)[0]
        if len(cols) > 0:
            row_medians.append(np.median(cols))
    
    
    for j in range(w):
        rows = np.where(mask[:, j] > 0)[0]
        if len(rows) > 0:
            col_medians.append(np.median(rows))
    
    
    cx = int(mode(row_medians, keepdims=True).mode[0])
    cy = int(mode(col_medians, keepdims=True).mode[0])
    
    
    return cx, cy

In [ ]:
# === 7. Fundus Radius Estimation (paper Algorithm 3) ===

def estimate_radius(mask, cx, cy):
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    if len(contours) == 0:
        return min(mask.shape) // 2
    

    largest = max(contours, key=cv2.contourArea)
    dists = []
    for p in largest:
        x, y = p[0]
        dists.append(np.sqrt((x-cx)**2 + (y-cy)**2))
    
    
    return int(mode(np.round(dists), keepdims=True).mode[0])

In [ ]:
# # === 8. Crop + Pad + Resize Standardization ===

# def standardize_fundus(img, size=512):
#     gray = apply_clahe(img)
#     mask = segment_foreground(gray)
    
    
#     cx, cy = estimate_center(mask)
#     r = estimate_radius(mask, cx, cy)
    
    
#     h, w = img.shape[:2]
    
    
#     x1 = max(cx - r, 0)
#     x2 = min(cx + r, w)
#     y1 = max(cy - r, 0)
#     y2 = min(cy + r, h)
    
    
#     cropped = img[y1:y2, x1:x2]
    
    
#     # pad to square
#     hh, ww = cropped.shape[:2]
#     pad_top = pad_bottom = pad_left = pad_right = 0
    
    
#     if hh > ww:
#         diff = hh - ww
#         pad_left = diff // 2
#         pad_right = diff - pad_left
#     else:
#         diff = ww - hh
#         pad_top = diff // 2
#         pad_bottom = diff - pad_top
    
    
#     padded = cv2.copyMakeBorder(
#     cropped, pad_top, pad_bottom, pad_left, pad_right,
#     cv2.BORDER_CONSTANT, value=0
#     )
    
    
#     standardized = cv2.resize(padded, (size, size), interpolation=cv2.INTER_AREA)
#     return standardized

In [ ]:
# === 8. Crop + Pad + Resize Standardization (FIXED) ===
def standardize_fundus(img, size=512):
    """
    Fixed version with proper error handling for edge cases
    """
    gray = apply_clahe(img)
    mask = segment_foreground(gray)
    
    cx, cy = estimate_center(mask)
    r = estimate_radius(mask, cx, cy)
    
    h, w = img.shape[:2]
    
    # Ensure radius is valid
    if r <= 0:
        r = min(h, w) // 4  # fallback radius
    
    # Calculate crop coordinates
    x1 = max(cx - r, 0)
    x2 = min(cx + r, w)
    y1 = max(cy - r, 0)
    y2 = min(cy + r, h)
    
    # Validate crop coordinates
    if x2 <= x1 or y2 <= y1:
        # Fallback: use center crop
        crop_size = min(h, w)
        y1 = max(0, (h - crop_size) // 2)
        y2 = y1 + crop_size
        x1 = max(0, (w - crop_size) // 2)
        x2 = x1 + crop_size
    
    cropped = img[y1:y2, x1:x2]
    
    # Double-check cropped image is valid
    if cropped.size == 0 or cropped.shape[0] == 0 or cropped.shape[1] == 0:
        # Ultimate fallback: resize original image
        print(f"Warning: Invalid crop, using original image")
        cropped = img
    
    # Pad to square
    hh, ww = cropped.shape[:2]
    pad_top = pad_bottom = pad_left = pad_right = 0
    
    if hh > ww:
        diff = hh - ww
        pad_left = diff // 2
        pad_right = diff - pad_left
    else:
        diff = ww - hh
        pad_top = diff // 2
        pad_bottom = diff - pad_top
    
    padded = cv2.copyMakeBorder(
        cropped, pad_top, pad_bottom, pad_left, pad_right,
        cv2.BORDER_CONSTANT, value=0
    )
    
    # Final validation before resize
    if padded.size == 0 or padded.shape[0] == 0 or padded.shape[1] == 0:
        # Return resized original as last resort
        return cv2.resize(img, (size, size), interpolation=cv2.INTER_AREA)
    
    standardized = cv2.resize(padded, (size, size), interpolation=cv2.INTER_AREA)
    return standardized

In [ ]:
# # === 9. Full Pipeline Wrapper ===

# def preprocess_fundus_image(path, size=512, show_steps=False):
#     img = cv2.imread(path)
#     img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    
#     if show_steps:
#         show(img, "Original")
    
    
#     out = standardize_fundus(img, size)
    
    
#     if show_steps:
#         show(out, "Standardized")
    
    
#     return out

In [ ]:
# === 9. Full Pipeline Wrapper (FIXED) ===
def preprocess_fundus_image(path, size=512, show_steps=False):
    """
    Fixed version with error handling for corrupted/missing images
    """
    try:
        img = cv2.imread(path)
        if img is None:
            raise ValueError(f"Failed to load image: {path}")
        
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        if show_steps:
            show(img, "Original")
        
        out = standardize_fundus(img, size)
        
        if show_steps:
            show(out, "Standardized")
        
        return out
    
    except Exception as e:
        print(f"Error processing {path}: {str(e)}")
        # Return a blank image as fallback
        return np.zeros((size, size, 3), dtype=np.uint8)

# Attention Unet Model

In [ ]:
# === CELL 5: Your Attention U-Net Model (Exact copy) ===

class AttentionBlock(nn.Module):
    def __init__(self, F_g, F_l, F_int):
        super().__init__()
        self.W_g = nn.Sequential(nn.Conv2d(F_g, F_int, 1), nn.BatchNorm2d(F_int))
        self.W_x = nn.Sequential(nn.Conv2d(F_l, F_int, 1), nn.BatchNorm2d(F_int))
        self.psi = nn.Sequential(nn.Conv2d(F_int, 1, 1), nn.BatchNorm2d(1), nn.Sigmoid())
        
    def forward(self, g, x):
        g1 = self.W_g(g)
        x1 = self.W_x(x)
        combined = F.relu(g1 + x1)
        attention_mask = self.psi(combined)
        return x * attention_mask

class ConvBlock(nn.Module):
    """Double convolution block"""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )
    
    def forward(self, x):
        return self.conv(x)

class UpConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.up = nn.Sequential(
            nn.Upsample(scale_factor=2),
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.up(x)

class AttentionUNet(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        
        # resnet = models.resnet50(weights='DEFAULT')
        resnet = models.resnet50(weights=None)
        
        # Encoder
        self.enc0 = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu)
        self.enc1 = resnet.maxpool
        self.enc2 = resnet.layer1
        self.enc3 = resnet.layer2
        self.enc4 = resnet.layer3
        self.center = resnet.layer4
        
        # Decoder with attention
        self.up4 = UpConv(2048, 1024)
        self.att4 = AttentionBlock(F_g=1024, F_l=1024, F_int=512)
        self.dec4 = ConvBlock(2048, 1024)
        
        self.up3 = UpConv(1024, 512)
        self.att3 = AttentionBlock(F_g=512, F_l=512, F_int=256)
        self.dec3 = ConvBlock(1024, 512)
        
        self.up2 = UpConv(512, 256)
        self.att2 = AttentionBlock(F_g=256, F_l=256, F_int=128)
        self.dec2 = ConvBlock(512, 256)
        
        self.up1 = UpConv(256, 64)
        self.att1 = AttentionBlock(F_g=64, F_l=64, F_int=32)
        self.dec1 = ConvBlock(128, 64)
        
        self.final = nn.Conv2d(64, num_classes, 1)
    
    def forward(self, x):
        # Encoder
        e0 = self.enc0(x)
        e1 = self.enc1(e0)
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)
        e4 = self.enc4(e3)
        c = self.center(e4)
        
        # Decoder with attention
        d4 = self.up4(c)
        e4_att = self.att4(g=d4, x=e4)
        d4 = torch.cat([d4, e4_att], dim=1)
        d4 = self.dec4(d4)
        
        d3 = self.up3(d4)
        e3_att = self.att3(g=d3, x=e3)
        d3 = torch.cat([d3, e3_att], dim=1)
        d3 = self.dec3(d3)
        
        d2 = self.up2(d3)
        e2_att = self.att2(g=d2, x=e2)
        d2 = torch.cat([d2, e2_att], dim=1)
        d2 = self.dec2(d2)
        
        d1 = self.up1(d2)
        e0_att = self.att1(g=d1, x=e0)
        d1 = torch.cat([d1, e0_att], dim=1)
        d1 = self.dec1(d1)
        
        out = self.final(d1)
        return out

In [ ]:
# === CELL 6: Load Your Pre-trained U-Net ===

# Load the trained U-Net model
unet_model = AttentionUNet(num_classes=3)
state_dict = torch.load(
    "/kaggle/input/models/aadityakafle/unet-model/pytorch/default/1/attention_unet_best.pt",
    map_location=torch.device("cpu")
)
unet_model.load_state_dict(state_dict)
unet_model = unet_model.to(device)
unet_model.eval()
print("✅ U-Net model loaded successfully!")

# Test the model on one image
fundus_path = "/kaggle/input/datasets/deathtrooper/multichannel-glaucoma-benchmark-dataset/full-fundus/full-fundus/REFUGE1-train-294.png"
test_img_processed = preprocess_fundus_image(fundus_path)

# Transform for U-Net
transform = augmentation.Compose([
    augmentation.Resize(128, 128),
    augmentation.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

test_tensor = transform(image=test_img_processed)['image'].unsqueeze(0).to(device)

with torch.no_grad():
    seg_output = unet_model(test_tensor)
    seg_output = F.interpolate(seg_output, size=(128, 128), mode='bilinear', align_corners=False)
    seg_pred = torch.argmax(seg_output, dim=1).squeeze().cpu().numpy()

print(f"✅ Segmentation test successful! Unique values in mask: {np.unique(seg_pred)}")

In [ ]:
import os
FUNDUS_DIR = "/kaggle/input/datasets/deathtrooper/multichannel-glaucoma-benchmark-dataset/full-fundus/full-fundus"

print(os.listdir(FUNDUS_DIR)[:20])  # first 20 files

In [ ]:
row = metadata.iloc[0]
print(os.path.join(root_dir, "full-fundus" + row['fundus']))


In [ ]:
# === CELL 7: Hybrid Dataset - RGB + Segmentation Masks for ViT ===

class HybridGlaucomaDataset(Dataset):
    """
    Dataset that combines:
    - Original RGB fundus image (3 channels)
    - U-Net predicted optic disc mask (1 channel)  
    - U-Net predicted optic cup mask (1 channel)
    Total: 5 channels for Vision Transformer
    """
    def __init__(self, metadata, root_dir, unet_model, img_size=224, device='cuda', is_train=False):
        self.metadata = metadata
        self.root_dir = root_dir
        self.unet_model = unet_model
        self.img_size = img_size
        self.device = device
        self.is_train = is_train
        
        # U-Net transform (same as your training)
        self.unet_transform = augmentation.Compose([
            augmentation.Resize(128, 128),  # Your U-Net was trained on 128x128
            augmentation.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
            ToTensorV2(),
        ])

        # Training augmentation (applied to RGB image only, before U-Net)
                # Inside HybridGlaucomaDataset.__init__, replace the augmentation block:
        if is_train:
            self.aug = augmentation.Compose([
                augmentation.HorizontalFlip(p=0.5),
                augmentation.VerticalFlip(p=0.5),
                augmentation.Rotate(limit=20, p=0.5),  # increased from 15
                augmentation.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.4),
                augmentation.GaussNoise(var_limit=(10.0, 50.0), p=0.2),
                augmentation.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=10, p=0.3),
                augmentation.ElasticTransform(alpha=1, sigma=50, p=0.2),  # NEW
                augmentation.GridDistortion(p=0.2),  # NEW
                augmentation.CoarseDropout(max_holes=8, max_height=16, max_width=16, p=0.2),  # NEW - acts like cutout
            ])
        else:
            self.aug = None
        
        # Normalization for 5-channel output
        self.normalize = transforms.Normalize(
            mean=[0.485, 0.456, 0.406, 0.5, 0.5],  # RGB + 2 masks
            std=[0.229, 0.224, 0.225, 0.5, 0.5]
        )
        
        self.unet_model.eval()
        print(f"✅ Dataset initialized with {len(self.metadata)} images")
    
    def __len__(self):
        return len(self.metadata)
    
    def __getitem__(self, idx):
        row = self.metadata.iloc[idx]
        
        # Load fundus image
        fundus_path = os.path.join(self.root_dir, "full-fundus" + row['fundus'])
        # image = np.array(Image.open(fundus_path).convert('RGB'))
        
        # Apply CLAHE preprocessing (same as your training)
        image = preprocess_fundus_image(fundus_path)

        # Apply augmentation BEFORE U-Net segmentation (training only)
        if self.aug is not None:
            image = self.aug(image=image)['image']
        
        # Resize to target size for ViT
        image_resized = cv2.resize(image, (self.img_size, self.img_size))
        image_array = image_resized.astype(np.float32) / 255.0
        
        # Get U-Net segmentation
        image_unet = self.unet_transform(image=image)['image'].unsqueeze(0).to(self.device)
        
        with torch.no_grad():
            seg_output = self.unet_model(image_unet)
            seg_output = F.interpolate(seg_output, size=(self.img_size, self.img_size), 
                                      mode='bilinear', align_corners=False)
            seg_pred = torch.argmax(seg_output, dim=1).squeeze(0).cpu().numpy()
        
        # Create binary masks (matching your mask encoding)
        disc_mask = (seg_pred == 2).astype(np.float32)  # OD = 2
        cup_mask = (seg_pred == 1).astype(np.float32)   # OC = 1
        
        # Stack 5 channels: R, G, B, Disc, Cup
        combined = np.stack([
            image_array[:, :, 0],  # R
            image_array[:, :, 1],  # G
            image_array[:, :, 2],  # B
            disc_mask,              # Disc mask
            cup_mask                # Cup mask
        ], axis=0).astype(np.float32)
        
        combined_tensor = torch.from_numpy(combined)
        combined_tensor = self.normalize(combined_tensor)


        label = int(row['types'])
            
        return combined_tensor, torch.tensor(label, dtype=torch.long)

print("✅ Hybrid dataset class defined")

In [ ]:
# === CELL 8: Vision Transformer Model ===

class VisionTransformerClassifier(nn.Module):
    """
    Vision Transformer modified to accept 5-channel input
    (RGB + Optic Disc Mask + Optic Cup Mask)
    Uses deit_small for better medical image performance
    """
    def __init__(self, num_classes=2, img_size=224, in_channels=5, 
                 pretrained=True, model_name='deit_small_patch16_224'):
        super(VisionTransformerClassifier, self).__init__()
        
        # Load pretrained ViT - USE SMALL instead of TINY
        self.vit = timm.create_model(model_name,
                                     pretrained=pretrained,
                                     num_classes=num_classes,
                                     drop_rate=0.3,       # increased from 0.2
                                     drop_path_rate=0.2    # increased from 0.1
                                    )
        
        # Modify input layer for 5 channels
        original_conv = self.vit.patch_embed.proj
        original_weight = original_conv.weight.data
        
        new_conv = nn.Conv2d(
            in_channels=in_channels,
            out_channels=original_conv.out_channels,
            kernel_size=original_conv.kernel_size,
            stride=original_conv.stride,
            padding=original_conv.padding,
            bias=original_conv.bias is not None
        )
        
        # Initialize weights
        with torch.no_grad():
            # Copy RGB weights
            new_conv.weight[:, :3, :, :] = original_weight
            # Initialize mask channels with average of RGB weights
            new_conv.weight[:, 3:, :, :] = original_weight.mean(dim=1, keepdim=True)
            if original_conv.bias is not None:
                new_conv.bias = original_conv.bias
        
        self.vit.patch_embed.proj = new_conv
        
        print(f"✅ ViT initialized: {model_name}")
        print(f"   Input: {in_channels} channels → {img_size}x{img_size}")
        print(f"   Output: {num_classes} classes")
    
    def forward(self, x):
        return self.vit(x)

print("✅ Vision Transformer model class defined")

In [ ]:
# === CELL 9: Prepare Data for Classification ===

print(f"\n📊 Label distribution:")
print(metadata['types'].value_counts())

In [ ]:
# # === CELL 10: Create Datasets and DataLoaders ===

# IMG_SIZE = 224
# BATCH_SIZE = 16

# # Create full dataset
# print("\n🔄 Creating hybrid dataset with U-Net predictions...")
# full_dataset = HybridGlaucomaDataset(
#     metadata=metadata,
#     root_dir=root_dir,
#     unet_model=unet_model,
#     img_size=IMG_SIZE,
#     device=device
# )

# # Split into train/val/test (70/15/15)
# total_size = len(full_dataset)
# train_size = int(0.70 * total_size)
# val_size = int(0.15 * total_size)
# test_size = total_size - train_size - val_size

# train_dataset, val_dataset, test_dataset = torch.utils.data.random_split(
#     full_dataset, [train_size, val_size, test_size],
#     generator=torch.Generator().manual_seed(42)
# )

# # Create DataLoaders
# train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, 
#                           shuffle=True, num_workers=0, pin_memory=True)
# val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, 
#                         shuffle=False, num_workers=0, pin_memory=True)
# test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, 
#                          shuffle=False, num_workers=0, pin_memory=True)

# print(f"\n✅ Data split complete:")
# print(f"   Train: {len(train_dataset)} samples")
# print(f"   Val:   {len(val_dataset)} samples")
# print(f"   Test:  {len(test_dataset)} samples")

# # Test dataloader
# test_batch = next(iter(train_loader))
# print(f"\n✅ DataLoader test:")
# print(f"   Batch input shape: {test_batch[0].shape}")  # Should be [batch, 5, 224, 224]
# print(f"   Batch labels shape: {test_batch[1].shape}")  # Should be [batch]

In [ ]:
# === UPDATED DATASET CREATION ===
IMG_SIZE = 224
BATCH_SIZE = 16

# Define your U-Net class here (AttentionUNet or whatever you used)
unet_model = AttentionUNet(num_classes=3)
model_path = "/kaggle/input/models/aadityakafle/unet-model/pytorch/default/1/attention_unet_best.pt"

state_dict = torch.load(model_path, map_location=device)
unet_model.load_state_dict(state_dict)
unet_model = unet_model.to(device)
unet_model.eval()

print("\n🔄 Creating hybrid datasets with augmentation...")

# Create SEPARATE train and validation datasets (with and without augmentation)
train_metadata = metadata.sample(frac=0.70, random_state=42)
remaining_metadata = metadata.drop(train_metadata.index)
val_metadata = remaining_metadata.sample(frac=0.5, random_state=42)
test_metadata = remaining_metadata.drop(val_metadata.index)

# Training dataset WITH augmentation
train_dataset = HybridGlaucomaDataset(
    metadata=train_metadata,
    root_dir=root_dir,
    unet_model=unet_model,
    img_size=IMG_SIZE,
    device=device,
    is_train=True  # Enable augmentation
)

# Validation dataset WITHOUT augmentation
val_dataset = HybridGlaucomaDataset(
    metadata=val_metadata,
    root_dir=root_dir,
    unet_model=unet_model,
    img_size=IMG_SIZE,
    device=device,
    is_train=False  # No augmentation
)

# Test dataset WITHOUT augmentation
test_dataset = HybridGlaucomaDataset(
    metadata=test_metadata,
    root_dir=root_dir,
    unet_model=unet_model,
    img_size=IMG_SIZE,
    device=device,
    is_train=False  # No augmentation
)

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, 
                          shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, 
                        shuffle=False, num_workers=0, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, 
                         shuffle=False, num_workers=0, pin_memory=True)

print(f"\n✅ Data split complete:")
print(f"   Train: {len(train_dataset)} samples (with augmentation)")
print(f"   Val:   {len(val_dataset)} samples (no augmentation)")
print(f"   Test:  {len(test_dataset)} samples (no augmentation)")

# Test dataloader
test_batch = next(iter(train_loader))
print(f"\n✅ DataLoader test:")
print(f"   Batch input shape: {test_batch[0].shape}")
print(f"   Batch labels shape: {test_batch[1].shape}")

In [ ]:
# === CELL 11: Initialize Vision Transformer (UPDATED) ===

vit_model = VisionTransformerClassifier(
    num_classes=2,
    img_size=IMG_SIZE,
    in_channels=5,
    pretrained=True,
    model_name='deit_small_patch16_224'  # CHANGED from deit_tiny
).to(device)

total_params = sum(p.numel() for p in vit_model.parameters())
trainable_params = sum(p.numel() for p in vit_model.parameters() if p.requires_grad)

print(f"\n📊 Model Statistics:")
print(f"   Total parameters: {total_params/1e6:.2f}M")
print(f"   Trainable parameters: {trainable_params/1e6:.2f}M")

In [ ]:
# === FOCAL LOSS FOR MEDICAL CLASSIFICATION ===
# Add this cell BEFORE the training configuration cell

class FocalLoss(nn.Module):
    """
    Focal Loss - focuses on hard-to-classify examples.
    Critical for medical imaging where missing a positive case (glaucoma) is costly.
    
    gamma: focusing parameter. Higher gamma = more focus on hard examples
           gamma=0 is equivalent to CrossEntropyLoss
           gamma=2 is the standard choice
    alpha: class weighting. Higher for the minority/important class
    """
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.gamma = gamma
        self.alpha = alpha  # tensor of shape [num_classes]
        self.reduction = reduction
    
    def forward(self, inputs, targets):
        # inputs: [batch_size, num_classes] (raw logits)
        # targets: [batch_size] (class indices)
        
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)  # probability of correct class
        
        focal_weight = (1 - pt) ** self.gamma
        
        if self.alpha is not None:
            alpha_t = self.alpha.to(inputs.device)[targets]
            focal_weight = alpha_t * focal_weight
        
        loss = focal_weight * ce_loss
        
        if self.reduction == 'mean':
            return loss.mean()
        elif self.reduction == 'sum':
            return loss.sum()
        return loss

print("✅ Focal Loss class defined")

In [ ]:
# ============================================================================
# CELL 12: Training Configuration (UPDATED for high recall)
# ============================================================================

NUM_EPOCHS = 50  # increased from 30, early stopping will handle it
LEARNING_RATE = 3e-5  # slightly lower for deit_small
WEIGHT_DECAY = 1e-2   # increased from 1e-3 for better regularization

# Alpha values for Focal Loss: higher weight for glaucoma class (class 1)
# Class 0 (non-glaucoma): 7549 samples
# Class 1 (glaucoma): 4767 samples
# We deliberately over-weight class 1 to boost recall
alpha_weights = torch.tensor([0.4, 0.6]).to(device)  # favor glaucoma detection

criterion = FocalLoss(alpha=alpha_weights, gamma=2.0)

optimizer = torch.optim.AdamW(
    vit_model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

# Cosine annealing with warm restarts for better convergence
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=10, T_mult=2, eta_min=1e-7
)

print(f"\n⚙️  Training Configuration:")
print(f"   Epochs: {NUM_EPOCHS}")
print(f"   Learning Rate: {LEARNING_RATE}")
print(f"   Weight Decay: {WEIGHT_DECAY}")
print(f"   Loss: Focal Loss (gamma=2.0, alpha={alpha_weights.tolist()})")
print(f"   Optimizer: AdamW")
print(f"   Scheduler: CosineAnnealingWarmRestarts")

In [ ]:
# ============================================================================
# CELL 13: Training Loop (UPDATED with recall/precision/F1 tracking)
# ============================================================================
from sklearn.metrics import recall_score, precision_score, f1_score, roc_auc_score

PATIENCE = 7  # increased slightly
best_val_f1 = 0.0  # track best F1 instead of just accuracy
best_val_recall = 0.0
patience_counter = 0

history = {
    'train_loss': [], 'val_loss': [], 
    'train_acc': [], 'val_acc': [],
    'val_recall': [], 'val_precision': [], 'val_f1': [], 'val_auc': []
}

print(f"\n🚀 Starting training...")
print("=" * 90)

for epoch in range(NUM_EPOCHS):
    # ===== TRAINING PHASE =====
    vit_model.train()
    train_loss = 0.0
    train_preds, train_labels = [], []
    
    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{NUM_EPOCHS} [Train]', leave=True)
    for inputs, labels in pbar:
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = vit_model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        
        # Gradient clipping to prevent exploding gradients
        torch.nn.utils.clip_grad_norm_(vit_model.parameters(), max_norm=1.0)
        
        optimizer.step()
        
        train_loss += loss.item()
        preds = torch.argmax(outputs, dim=1)
        train_preds.extend(preds.cpu().numpy())
        train_labels.extend(labels.cpu().numpy())
        
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    train_loss /= len(train_loader)
    train_acc = accuracy_score(train_labels, train_preds)
    
    # ===== VALIDATION PHASE =====
    vit_model.eval()
    val_loss = 0.0
    val_preds, val_labels, val_probs = [], [], []
    
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = vit_model(inputs)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item()
            
            # Get probabilities for AUC and threshold tuning
            probs = torch.softmax(outputs, dim=1)[:, 1]  # probability of glaucoma
            val_probs.extend(probs.cpu().numpy())
            
            preds = torch.argmax(outputs, dim=1)
            val_preds.extend(preds.cpu().numpy())
            val_labels.extend(labels.cpu().numpy())
    
    val_loss /= len(val_loader)
    val_acc = accuracy_score(val_labels, val_preds)
    
    # === COMPUTE DETAILED METRICS ===
    val_recall = recall_score(val_labels, val_preds, pos_label=1, zero_division=0)
    val_precision = precision_score(val_labels, val_preds, pos_label=1, zero_division=0)
    val_f1 = f1_score(val_labels, val_preds, pos_label=1, zero_division=0)
    try:
        val_auc = roc_auc_score(val_labels, val_probs)
    except:
        val_auc = 0.0
    
    # Save history
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    history['val_recall'].append(val_recall)
    history['val_precision'].append(val_precision)
    history['val_f1'].append(val_f1)
    history['val_auc'].append(val_auc)
    
    # Print epoch summary
    print(f'Epoch {epoch+1:02d}/{NUM_EPOCHS} | '
          f'Train Loss: {train_loss:.4f} | Train Acc: {train_acc*100:.1f}% | '
          f'Val Loss: {val_loss:.4f} | Val Acc: {val_acc*100:.1f}% | '
          f'Recall: {val_recall:.4f} | Prec: {val_precision:.4f} | '
          f'F1: {val_f1:.4f} | AUC: {val_auc:.4f}')
    
    # ===== SAVE BEST MODEL BASED ON F1 SCORE (balances recall & precision) =====
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_val_recall = val_recall
        patience_counter = 0
        torch.save(vit_model.state_dict(), 'vit_classifier_best.pt')
        print(f'  ✅ Best model saved! F1: {val_f1:.4f} | Recall: {val_recall:.4f}')
    else:
        patience_counter += 1
        print(f'  ⚠️  No improvement. Patience: {patience_counter}/{PATIENCE}')
    
    if patience_counter >= PATIENCE:
        print(f'\n🛑 Early stopping at epoch {epoch+1}!')
        print(f'   Best F1: {best_val_f1:.4f} | Best Recall: {best_val_recall:.4f}')
        break
    
    scheduler.step(epoch + val_loss / (val_loss + 1))  # for warm restarts

print("=" * 90)
print(f'✅ Training complete!')
print(f'   Best F1: {best_val_f1:.4f}')
print(f'   Best Recall: {best_val_recall:.4f}')

In [ ]:
import joblib
joblib.dump(vit_model, 'vit_classifier_best.pt')

In [ ]:
# ===========================================================================
# THRESHOLD OPTIMIZATION - Critical for medical classification
# ============================================================================
# Instead of using argmax (threshold=0.5), find the optimal threshold
# that maximizes recall while keeping precision acceptable

from sklearn.metrics import precision_recall_curve

# Load best model
vit_model.load_state_dict(torch.load('vit_classifier_best.pt'))
vit_model.eval()

# Collect all validation probabilities
all_probs = []
all_labels = []

with torch.no_grad():
    for inputs, labels in val_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = vit_model(inputs)
        probs = torch.softmax(outputs, dim=1)[:, 1]
        all_probs.extend(probs.cpu().numpy())
        
        all_labels.extend(labels.cpu().numpy())

all_probs = np.array(all_probs)
all_labels = np.array(all_labels)

# Find optimal threshold
precisions, recalls, thresholds = precision_recall_curve(all_labels, all_probs)

# Strategy: Find threshold where recall >= 0.90 and precision is maximized
best_threshold = 0.5
best_f1_at_high_recall = 0.0

for i in range(len(thresholds)):
    if recalls[i] >= 0.90:  # We want at least 90% recall
        f1 = 2 * (precisions[i] * recalls[i]) / (precisions[i] + recalls[i] + 1e-8)
        if f1 > best_f1_at_high_recall:
            best_f1_at_high_recall = f1
            best_threshold = thresholds[i]

print(f"\n🎯 Optimal Threshold Analysis:")
print(f"   Default threshold (0.5):")
default_preds = (all_probs >= 0.5).astype(int)
print(f"     Recall:    {recall_score(all_labels, default_preds, pos_label=1):.4f}")
print(f"     Precision: {precision_score(all_labels, default_preds, pos_label=1):.4f}")
print(f"     F1:        {f1_score(all_labels, default_preds, pos_label=1):.4f}")

print(f"\n   Optimized threshold ({best_threshold:.4f}):")
optimized_preds = (all_probs >= best_threshold).astype(int)
print(f"     Recall:    {recall_score(all_labels, optimized_preds, pos_label=1):.4f}")
print(f"     Precision: {precision_score(all_labels, optimized_preds, pos_label=1):.4f}")
print(f"     F1:        {f1_score(all_labels, optimized_preds, pos_label=1):.4f}")
print(f"     AUC:       {roc_auc_score(all_labels, all_probs):.4f}")

OPTIMAL_THRESHOLD = best_threshold
print(f"\n✅ Use OPTIMAL_THRESHOLD = {OPTIMAL_THRESHOLD:.4f} for inference")

In [ ]:
# === PLOT TRAINING HISTORY ===
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Loss plot
ax1.plot(history['train_loss'], label='Train Loss', marker='o')
ax1.plot(history['val_loss'], label='Val Loss', marker='s')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training and Validation Loss')
ax1.legend()
ax1.grid(True)

# Accuracy plot
ax2.plot(history['train_acc'], label='Train Acc', marker='o')
ax2.plot(history['val_acc'], label='Val Acc', marker='s')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Training and Validation Accuracy')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.savefig('training_history.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n📊 Training history plot saved as 'training_history.png'")


In [ ]:
# ============================================================================
# INFERENCE & FULL METRICS EVALUATION
# ============================================================================
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, recall_score, precision_score, f1_score,
    roc_auc_score, roc_curve, precision_recall_curve,
    confusion_matrix, classification_report, average_precision_score
)
from tqdm import tqdm

# ============================================================================
# 1. LOAD MODEL & RUN INFERENCE
# ============================================================================

# Load best saved model
# vit_model.load_state_dict(torch.load('vit_classifier_best.pt', map_location=device, weights_only=False))
vit_model.eval()

all_preds = []
all_labels = []
all_probs = []

print("🔍 Running inference on test set...")
with torch.no_grad():
    for inputs, labels in tqdm(test_loader, desc="Inference"):
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = vit_model(inputs)

        probs = torch.softmax(outputs, dim=1)[:, 1]   # P(glaucoma)
        preds = torch.argmax(outputs, dim=1)

        all_probs.extend(probs.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs  = np.array(all_probs)

# ============================================================================
# 2. THRESHOLD TUNING — find threshold that maximises F1 on test set
# ============================================================================

thresholds = np.arange(0.1, 0.9, 0.01)
f1_scores  = [f1_score(all_labels, (all_probs >= t).astype(int), zero_division=0)
              for t in thresholds]

best_thresh = thresholds[np.argmax(f1_scores)]
best_preds  = (all_probs >= best_thresh).astype(int)

print(f"\n🎯 Best threshold (max F1): {best_thresh:.2f}")

# ============================================================================
# 3. CORE METRICS — default (0.5) vs best threshold
# ============================================================================

def print_metrics(labels, preds, probs, label=""):
    acc       = accuracy_score(labels, preds)
    recall    = recall_score(labels, preds, pos_label=1, zero_division=0)
    precision = precision_score(labels, preds, pos_label=1, zero_division=0)
    f1        = f1_score(labels, preds, pos_label=1, zero_division=0)
    specificity = recall_score(labels, preds, pos_label=0, zero_division=0)  # TN / (TN+FP)
    try:
        auc = roc_auc_score(labels, probs)
        ap  = average_precision_score(labels, probs)
    except:
        auc = ap = 0.0

    print(f"\n{'='*55}")
    print(f"  {label}")
    print(f"{'='*55}")
    print(f"  Accuracy        : {acc:.4f}  ({acc*100:.2f}%)")
    print(f"  Recall (Sens.)  : {recall:.4f}  ← % glaucoma caught")
    print(f"  Specificity     : {specificity:.4f}  ← % healthy correct")
    print(f"  Precision       : {precision:.4f}")
    print(f"  F1 Score        : {f1:.4f}")
    print(f"  ROC-AUC         : {auc:.4f}")
    print(f"  Avg Precision   : {ap:.4f}  (PR-AUC)")
    print(f"{'='*55}")
    return {'acc': acc, 'recall': recall, 'specificity': specificity,
            'precision': precision, 'f1': f1, 'auc': auc, 'ap': ap}

print_metrics(all_labels, all_preds,  all_probs, "DEFAULT THRESHOLD (0.50)")
print_metrics(all_labels, best_preds, all_probs, f"BEST THRESHOLD    ({best_thresh:.2f})")

# Full sklearn report
print("\n📋 Classification Report (best threshold):")
print(classification_report(all_labels, best_preds,
                             target_names=['Healthy', 'Glaucoma']))

# ============================================================================
# 4. PLOTS
# ============================================================================

fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.suptitle("Model Evaluation — Full Metrics", fontsize=16, fontweight='bold')

# ── (A) Confusion Matrix ────────────────────────────────────────────────────
cm = confusion_matrix(all_labels, best_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0, 0],
            xticklabels=['Healthy', 'Glaucoma'],
            yticklabels=['Healthy', 'Glaucoma'])
tn, fp, fn, tp = cm.ravel()
axes[0, 0].set_title(f'Confusion Matrix (thresh={best_thresh:.2f})\n'
                     f'TP={tp}  FP={fp}  FN={fn}  TN={tn}')
axes[0, 0].set_ylabel('True Label')
axes[0, 0].set_xlabel('Predicted Label')

# ── (B) ROC Curve ───────────────────────────────────────────────────────────
fpr, tpr, _ = roc_curve(all_labels, all_probs)
auc_score    = roc_auc_score(all_labels, all_probs)
axes[0, 1].plot(fpr, tpr, 'b-', lw=2, label=f'AUC = {auc_score:.4f}')
axes[0, 1].plot([0, 1], [0, 1], 'k--', lw=1)
axes[0, 1].fill_between(fpr, tpr, alpha=0.1)
axes[0, 1].set_title('ROC Curve')
axes[0, 1].set_xlabel('False Positive Rate')
axes[0, 1].set_ylabel('True Positive Rate (Recall)')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# ── (C) Precision-Recall Curve ──────────────────────────────────────────────
prec_curve, rec_curve, _ = precision_recall_curve(all_labels, all_probs)
ap_score = average_precision_score(all_labels, all_probs)
axes[0, 2].plot(rec_curve, prec_curve, 'g-', lw=2, label=f'AP = {ap_score:.4f}')
axes[0, 2].fill_between(rec_curve, prec_curve, alpha=0.1, color='green')
axes[0, 2].set_title('Precision-Recall Curve')
axes[0, 2].set_xlabel('Recall')
axes[0, 2].set_ylabel('Precision')
axes[0, 2].legend()
axes[0, 2].grid(True, alpha=0.3)

# ── (D) F1 vs Threshold ─────────────────────────────────────────────────────
axes[1, 0].plot(thresholds, f1_scores, 'purple', lw=2)
axes[1, 0].axvline(best_thresh, color='red', linestyle='--',
                   label=f'Best thresh={best_thresh:.2f}')
axes[1, 0].set_title('F1 Score vs Decision Threshold')
axes[1, 0].set_xlabel('Threshold')
axes[1, 0].set_ylabel('F1 Score')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# ── (E) Recall & Precision vs Threshold ─────────────────────────────────────
rec_t   = [recall_score(all_labels, (all_probs >= t).astype(int), zero_division=0)
           for t in thresholds]
prec_t  = [precision_score(all_labels, (all_probs >= t).astype(int), zero_division=0)
           for t in thresholds]
axes[1, 1].plot(thresholds, rec_t,  'b-',  lw=2, label='Recall (Sensitivity)')
axes[1, 1].plot(thresholds, prec_t, 'r-',  lw=2, label='Precision')
axes[1, 1].axvline(best_thresh, color='gray', linestyle='--',
                   label=f'Best thresh={best_thresh:.2f}')
axes[1, 1].set_title('Recall & Precision vs Threshold')
axes[1, 1].set_xlabel('Threshold')
axes[1, 1].set_ylabel('Score')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

# ── (F) Predicted Probability Distribution ──────────────────────────────────
healthy  = all_probs[all_labels == 0]
glaucoma = all_probs[all_labels == 1]
axes[1, 2].hist(healthy,  bins=40, alpha=0.6, color='blue',  label='Healthy')
axes[1, 2].hist(glaucoma, bins=40, alpha=0.6, color='red',   label='Glaucoma')
axes[1, 2].axvline(best_thresh, color='black', linestyle='--',
                   label=f'Thresh={best_thresh:.2f}')
axes[1, 2].set_title('Predicted Probability Distribution')
axes[1, 2].set_xlabel('P(Glaucoma)')
axes[1, 2].set_ylabel('Count')
axes[1, 2].legend()
axes[1, 2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('inference_metrics.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n✅ Plot saved to inference_metrics.png")

In [ ]:
vit_model.load_state_dict(torch.load('vit_classifier_best.pt', map_location=device, weights_only=False, mmap=True))

In [ ]:
import shutil
shutil.copy('vit_classifier_best.pt', '/kaggle/working/vit_classifier_best.pt')
vit_model.load_state_dict(torch.load('/kaggle/working/vit_classifier_best.pt', map_location=device, weights_only=False))

In [ ]:
import os
# Search all common Kaggle paths
for root, dirs, files in os.walk('/kaggle'):
    for f in files:
        if f.endswith('.pt'):
            full = os.path.join(root, f)
            print(f"{full}  —  {os.path.getsize(full)/1e6:.2f} MB")

In [ ]:
import torch

# ── Fix UNet ──────────────────────────────────────────────────────────────
unet_sd = torch.load("/kaggle/input/models/aryanbhattarai/attention-unet-best/pytorch/default/1/attention_unet_best.pt")
unet_cpu = {k: v.cpu() for k, v in unet_sd.items()}
torch.save(unet_cpu, "/kaggle/working/attention_unet_best_cpu.pt")
print("✅ UNet saved")

# ── Fix ViT ───────────────────────────────────────────────────────────────
vit_sd = torch.load("/kaggle/input/models/aryanbhattarai/vit-classifier-best/pytorch/default/1/vit_classifier_best.pt")
vit_cpu = {k: v.cpu() for k, v in vit_sd.items()}
torch.save(vit_cpu, "/kaggle/working/vit_classifier_best_cpu.pt")
print("✅ ViT saved")